In [ ]:
"""
Advanced Model Evaluation Module
ROC Curve + PR Curve + Confusion Matrix + Metrics
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================
# PATH CONFIG (PRO PRECISE)
# =========================

BASE_DIR = Path.cwd().parent  # notebook → root project

MODEL_DIR = BASE_DIR / "model"
DATA_DIR = BASE_DIR / "data" / "processed"
REPORT_DIR = MODEL_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# LOAD MODEL
# =========================

model_path = MODEL_DIR / "model.pkl"

print("MODEL PATH:", model_path)
print("EXISTS:", model_path.exists())

model = joblib.load(model_path)

ModuleNotFoundError: No module named 'probabilities'

In [ ]:

# =========================
# LOAD MODEL + DATA
# =========================
def load_model_and_data():
    model = joblib.load(MODEL_DIR / "model.pkl")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

    X_test = test_df.drop("Outcome", axis=1)
    y_test = test_df["Outcome"]

    print(f"Model loaded")
    print(f"Test samples: {X_test.shape[0]}")

    return model, X_test, y_test



In [7]:

# =========================
# CONFUSION MATRIX
# =========================
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    path = REPORT_DIR / "confusion_matrix.png"
    plt.savefig(path, dpi=300)
    plt.close()

    print(f"Saved: {path}")



In [8]:

# =========================
# ROC CURVE
# =========================
def plot_roc(y_true, y_proba):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")

    plt.title("ROC Curve")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()

    path = REPORT_DIR / "roc_curve.png"
    plt.savefig(path, dpi=300)
    plt.close()

    print(f"Saved: {path}")

    return roc_auc


In [14]:

# =========================
# PRECISION RECALL
# =========================
def plot_pr(y_true, y_proba):
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    pr_auc = auc(recall, precision)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AUC = {pr_auc:.3f}")

    plt.title("Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()

    path = REPORT_DIR / "pr_curve.png"
    plt.savefig(path, dpi=300)
    plt.close()

    print(f"Saved: {path}")

    return pr_auc


In [ ]:

# =========================
# MAIN EVALUATION
# =========================
def evaluate():
    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)

    model, X_test, y_test = load_model_and_data()

    y_pred = model.predict(X_test)

    # probabilités (safe)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = None
    threshold = 0.3  # important pour diabète (plus sensible)
    y_pred = (y_proba >= threshold).astype(int)
    
    # ================= metrics =================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("\nMetrics Summary:")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1-score :", f1_score(y_test, y_pred))

    # ================= plots =================
    plot_confusion_matrix(y_test, y_pred)

    if y_proba is not None:
        roc_auc = plot_roc(y_test, y_proba)
        pr_auc = plot_pr(y_test, y_proba)

        print("\nROC AUC:", roc_auc)
        print("PR AUC :", pr_auc)

    print("\nEvaluation completed ")


if __name__ == "__main__":
    evaluate()

MODEL EVALUATION
Model loaded ✔
Test samples: 154

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.78      0.80       100
           1       0.63      0.69      0.65        54

    accuracy                           0.75       154
   macro avg       0.72      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154


Metrics Summary:
Accuracy : 0.7467532467532467
Precision: 0.6271186440677966
Recall   : 0.6851851851851852
F1-score : 0.6548672566371682
Saved: c:\Users\ThinkPad\OneDrive\Bureau\ai-prediction-system\model\reports\confusion_matrix.png
Saved: c:\Users\ThinkPad\OneDrive\Bureau\ai-prediction-system\model\reports\roc_curve.png
Saved: c:\Users\ThinkPad\OneDrive\Bureau\ai-prediction-system\model\reports\pr_curve.png

ROC AUC: 0.8266666666666667
PR AUC : 0.7045851936180234

Evaluation completed 
